# Experiment 4.0.5 — Temporal resolution vs event capacity

Analysis-only notebook. Training is performed by the 45-task Slurm array. This notebook reads finalized artifacts only.

Primary comparisons:
- `repeat4 - fixed250`: more internal SNN timesteps without adding within-bin information.
- `raw64 - repeat4`: value of genuine within-bin event timing.
- Binary vs Multi-H vs Multi-HO across each representation.
- Output WholeCount vs Hidden WholeCount + Linear vs Uend + Linear.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
root = repo_root / 'notebooks' / 'artifacts' / 'experiment_4_0_5_temporal_resolution_event_capacity' / 'hierarchical_temporal_resolution_event_cap_v1'
runs_file = root / 'runs.csv'
summary_file = root / 'summary.csv'
effects_file = root / 'paired_effects.csv'
print('Artifacts:', root)
print('runs.csv exists:', runs_file.exists())

In [ ]:
if not runs_file.exists():
    print('Finalized artifacts are not available yet. Run the Slurm array and finalizer first.')
else:
    runs = pd.read_csv(runs_file)
    test = runs[runs['split'] == 'test'].copy()
    display(test[['representation','variant','seed','valid_count_balanced_accuracy','hidden_count_linear_balanced_accuracy','uend_linear_balanced_accuracy']].sort_values(['representation','variant','seed']))

In [ ]:
if summary_file.exists():
    summary = pd.read_csv(summary_file)
    display(summary)

In [ ]:
if runs_file.exists():
    test = pd.read_csv(runs_file).query("split == 'test'")
    plot_rows = []
    for (representation, variant), group in test.groupby(['representation','variant']):
        for readout, column in {
            'Output WholeCount': 'valid_count_balanced_accuracy',
            'Hidden Count + Linear': 'hidden_count_linear_balanced_accuracy',
            'Uend + Linear': 'uend_linear_balanced_accuracy',
        }.items():
            plot_rows.append({
                'representation': representation,
                'variant': variant,
                'readout': readout,
                'mean_ba': group[column].mean(),
                'std_ba': group[column].std(),
            })
    plot_df = pd.DataFrame(plot_rows)
    display(plot_df)

In [ ]:
if effects_file.exists():
    effects = pd.read_csv(effects_file)
    display(effects.groupby(['effect','readout'], dropna=False)['delta_balanced_accuracy'].agg(['mean','std','count']).reset_index())

## Interpretation guide

1. If `repeat4 > fixed250`, more binary/multi-event firing opportunities help even without adding information.
2. If `raw64 > repeat4`, genuine within-bin timing adds discriminative information.
3. If `Multi-HO - Binary` shrinks from Fixed250 to Repeat4/Raw64, multi-event communication is partly compensating for coarse temporal discretization.
4. If `Uend + Linear` stays high while Output WholeCount improves with temporal resolution, the main gain is better state-to-spike export rather than better endpoint memory.